# 스모크 테스트 v2 — 3층 평가 구조 확인

깨끗한 순서로 한 번에. **SMOKE 모드**(코퍼스 5천 · N=3 · seed 1개)로 가볍게 3층 구조만 확인.

**순서:** 0~2 실행 → 3에서 런타임 재시작 → 4부터 이어서.

⚠️ 자꾸 런타임이 종료되면 Colab 사용량 한도일 수 있음 → 좀 쉬었다 새 세션으로.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 코드 받기 (HY, 강제 정렬 — 로컬 수정 있어도 pull 안 막힘)

In [ ]:
import os
%cd /content
if os.path.isdir('knowledge-rag-qa'):
    %cd knowledge-rag-qa
    !git fetch origin HY && git reset --hard origin/HY
else:
    !git clone -b HY https://github.com/likelion-nlp-project2/knowledge-rag-qa.git
    %cd knowledge-rag-qa
!git log --oneline -1
print('--- data/ 코퍼스 파일 (ko_miracl_subset.jsonl 있어야 함) ---')
!ls -la data/

## 2. 설치 + bitsandbytes(CUDA13) 수정
torch/numpy는 코랩 기본 버전으로 **고정(==)**해 torchvision/torchaudio 충돌 방지.

In [ ]:
import importlib.metadata as md
pinned = f"torch=={md.version('torch')} numpy=={md.version('numpy')}"
print('고정:', pinned)
!pip -q install -U {pinned} datasets chromadb sentence-transformers transformers accelerate bitsandbytes ragas langchain-community
# bitsandbytes가 찾는 CUDA13 라이브러리 제공 + 로더 경로에 링크
!pip install -q nvidia-nvjitlink-cu13
!ln -sf $(find / -name 'libnvJitLink.so.13' 2>/dev/null | head -1) /usr/lib/x86_64-linux-gnu/ 2>/dev/null
!ldconfig
print('설치 완료 — 다음 셀(3) 안내대로 런타임 재시작')

## 3. ⚠️ 런타임 재시작 (필수)
메뉴 → **런타임 → 세션 다시 시작**. 그다음 **0~2번은 다시 실행하지 말고 4번부터** 이어서.
(클론된 파일·설치·심볼릭 링크는 재시작해도 유지됨. 커널 메모리만 초기화.)

## 4. bitsandbytes 확인 (가벼움 — 여기서 죽으면 CUDA 문제)

In [ ]:
import bitsandbytes as bnb
print('bnb OK:', bnb.__version__)

## 5. 검색 평가 (Qwen 없이 — 가벼운 체크포인트). 여기 성공하면 코드/데이터 정상

In [ ]:
import os
os.environ['SMOKE'] = '1'   # 코퍼스 5천 · N=3 · seed 1개
%cd /content/knowledge-rag-qa
%run evaluation/run_retrieval_eval.py

## 6. 생성 3층 평가 (SMOKE) — 1층 규칙지표 → 2층 RAGAS → 3층 사람시트

In [ ]:
%cd /content/knowledge-rag-qa
%run evaluation/run_generation_eval.py

## 7. 결과 — 사람 채점 시트 확인

In [ ]:
import pandas as pd
pd.read_csv('evaluation/data/human_check.csv')